<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## Step Size and Convergence

**Compare convergence, oscillation, and divergence by changing only the step size.**

Keep the four clock observations, corrected-error calculation, and squared-error objective. Remove the calibration equality and correction bound; the settings are unrestricted unless the rate is explicitly held at zero. Hold the rate at zero and start the correction at zero in every comparison.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · Keep the decision and objective fixed

The observed clock errors are $d=(-2,-1,-1,0)$ min at times $t=(0,1,2,3)$ h. For observation $i=1,\ldots,4$, the corrected error is

$$
y_i=d_i+q+rt_i.
$$

The decision $x=(q,r)$ contains the initial correction $q$ in min and rate correction $r$ in min/h. The response vector is $y=\operatorname{Sim}(x)$. The scalar objective is the sum of squared errors:

$$
f(x)=\sum_{i=1}^{4}(d_i+q+rt_i)^2.
$$

We write $f(x)$ for the objective after composing the response calculation with the squared-error score. Its unit is $\mathrm{min}^2$. Every finite pair of real settings is feasible in this formulation. A smaller score therefore gives a better candidate.

This is a smooth, unconstrained least-squares problem and a quadratic program (QP). These names describe the problem form; gradient descent and Newton's method are solution procedures.

In the calculations, coordinates are numerical values in the stated units: $q=1$ means 1 min, and $r=1$ means 1 min/h. Gradient sizes and step sizes refer to this fixed coordinate convention. Changing units changes the coordinate scaling and can require a different step size.

With $r=0$ held fixed, the one-setting objective is

$$
f(q,0)=(q-2)^2+2(q-1)^2+q^2=4(q-1)^2+2.
$$

This identity gives the exact reference $q^\star=1$ min and $f(q^\star,0)=2\ \mathrm{min}^2$. Every numerical candidate can be compared with this exact reference. The observed data and response calculation remain fixed as the candidate changes.


In [ ]:
import numpy as np

# Fixed observations: each error corresponds to the time at the same position.
OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min
OBSERVATION_TIMES = np.array([0.0, 1.0, 2.0, 3.0])   # h


def simulate_clock(decision):
    correction, rate = np.asarray(decision, dtype=float)
    return OBSERVED_ERRORS + correction + rate * OBSERVATION_TIMES


def objective_function(response):
    return float(np.dot(response, response))


def evaluate_clock(decision):
    decision = np.asarray(decision, dtype=float)
    if decision.shape != (2,) or not np.isfinite(decision).all():
        raise ValueError("Use two finite numerical settings: correction and rate.")
    response = simulate_clock(decision)
    gradient = 2.0 * np.array([response.sum(), OBSERVATION_TIMES @ response])
    return {"decision": decision.copy(), "response": response,
            "feasible": True, "objective": objective_function(response),
            "gradient": gradient,
            "gradient_norm": float(np.linalg.norm(gradient))}


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The offset derivative is $8(q-1)$. With iteration index $k$, initial correction $q^{(0)}=0$, and positive step-size hyperparameter $\alpha$, the recurrence is

$$
q^{(k+1)}=q^{(k)}-\alpha\,8(q^{(k)}-1).
$$

The same recurrence generates every trace below. Changing $\alpha$ changes the proposed settings, not the physical response of a fixed setting.


In [ ]:
def offset_history(step_size, steps, initial_correction=0.0):
    correction = float(initial_correction)
    history = [correction]
    for _ in range(steps):
        slope = 8.0 * (correction - 1.0)
        correction -= step_size * slope
        history.append(correction)
    return np.array(history)


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Change only the step size

Keep $q^{(0)}=0$, $r=0$, and the same objective. Only $\alpha$ changes.

| Step size $\alpha$ | First settings $q^{(0)},q^{(1)},q^{(2)}$ | Result |
|:---|:---|:---|
| 0.05 | $0,\ 0.4,\ 0.64$ | Moves toward 1 gradually |
| 0.125 | $0,\ 1,\ 1$ | Reaches the reference in one update |
| 0.25 | $0,\ 2,\ 0$ | Oscillates without improvement in score |
| 0.30 | $0,\ 2.4,\ -0.96$ | Alternates sides and moves farther away |

Subtracting 1 from the update gives one short explanation of all four cases:

$$
q^{(k+1)}-1=(1-8\alpha)(q^{(k)}-1).
$$

The distance to 1 shrinks when $|1-8\alpha|<1$, or $0<\alpha<0.25$. At $\alpha=0.25$ the error only changes sign. At $\alpha=0.30$ it is multiplied by $-1.4$, so its magnitude grows. These limits belong to this one-setting quadratic in these units; they are not universal learning-rate rules.

Compare the curves against the same horizontal reference. A direction that improves the objective for a sufficiently small step can still fail when the step is too large.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-2_iterative_optimization_gradient_descent_and_newton_method/assets/step_size_comparison.svg" alt="Four runs from correction zero keep the offset-only objective fixed. Step sizes 0.05 and 0.125 approach or reach correction 1; 0.25 oscillates between 0 and 2; 0.30 diverges while alternating sides." width="520" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: white;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The numerical traces use the same recurrence for every step size.

In [ ]:
for step_size in (0.05, 0.125, 0.25, 0.30):
    print(f"alpha={step_size:.3f}: {np.round(offset_history(step_size, 4), 4)}")

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · One hyperparameter changes the trajectory

The interactive comparison keeps the offset-only restriction $r=0$ and starting correction $q^{(0)}=0$. The observations and objective stay fixed. The slider changes only $\alpha$, and the plot always shows eight updates. The orange marker is the final displayed candidate; the teal line is the exact reference $q=1$ min.

At $\alpha=0.05$ the candidates approach 1; at 0.125 they reach it in one update; at 0.25 they alternate between 0 and 2; above 0.25 they move farther away. The plot rescales to include the entire trace, so compare the axis values as well as the line shape. Eight displayed updates are a fixed demonstration budget, not a declaration of convergence.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The plotting definitions use a Matplotlib slider to change the step size and recompute the eight updates.


In [ ]:
import sys
import matplotlib

BLUE, TEAL, ORANGE, PURPLE = "#2878b5", "#168578", "#e78b24", "#8854a5"


def get_pyplot(interactive=False):
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (RuntimeError, ValueError):
            from matplotlib.backends import backend_registry
            backend_registry._clear()
            matplotlib.use("widget", force=True)
    import matplotlib.pyplot as plt
    return plt


In [ ]:
def show_step_size_slider():
    plt = get_pyplot(interactive=True)
    from matplotlib.widgets import Slider

    figure, axis = plt.subplots(figsize=(5.6, 4.6))
    figure.subplots_adjust(left=0.15, right=0.96, top=0.87, bottom=0.36)
    steps = np.arange(9)
    trace = offset_history(0.05, 8)
    path, = axis.plot(steps, trace, "o-", color=BLUE, label="Proposed settings")
    final, = axis.plot([8], [trace[-1]], "o", color=ORANGE, markersize=8)
    axis.axhline(1.0, color=TEAL, linestyle="--", label="Exact reference: 1 min")
    axis.set(xlabel="Number of updates k", ylabel="Correction q (min)",
             xticks=[0, 2, 4, 6, 8])
    axis.set_title("Step size changes the same eight updates", fontsize=11)
    axis.grid(alpha=0.25)
    axis.legend(fontsize=10, loc="upper left")
    status = figure.text(0.15, 0.19, "", fontsize=10)
    slider_axis = figure.add_axes([0.20, 0.08, 0.60, 0.04])
    slider = Slider(slider_axis, "alpha", 0.01, 0.30, valinit=0.05,
                    valstep=0.005, valfmt="%.3f", color=PURPLE)
    slider.label.set_color(PURPLE)

    def update(_):
        trace = offset_history(slider.val, 8)
        path.set_ydata(trace)
        final.set_ydata([trace[-1]])
        low, high = min(trace.min(), 1.0), max(trace.max(), 1.0)
        margin = max(0.2, 0.20 * (high - low))
        axis.set_ylim(low - margin, high + 2.0 * margin)
        score = evaluate_clock((trace[-1], 0.0))["objective"]
        status.set_text(f"After 8 updates: q = {trace[-1]:.3f} min; f = {score:.3f} min²")
        figure.canvas.draw_idle()

    slider.on_changed(update)
    figure._sliders = [slider]
    figure._update_step_size = update
    update(None)
    return figure

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The initial view uses $\alpha=0.05$. Each slider change recomputes all eight candidates from zero.

In [ ]:
step_size_figure = show_step_size_slider()
get_pyplot().show()

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · Source material and scope

The supplied `Lecture_9_2_Traditional_Optimization02/9_2.pptx`, slides 4–8, introduces gradient-based methods, steepest descent, Newton's use of the Hessian, and initialization. Its `Example_9_2/hands_on_1.py` supplies a repeated negative-gradient update, a gradient-norm stopping test, and an iteration limit. This notebook isolates the effect of step size with numerical traces and a slider.

The source example's clipping operation is omitted because these settings are unrestricted. Plain gradient and Newton updates do not enforce a calibration equality or correction bound; restoring those requirements would require a constraint-aware method and explicit feasibility checks.
